In [0]:
from pyspark.sql import functions as F

staging_orders = spark.table("brazilian_ecommerce.prod.staging_orders")
staging_customers = spark.table("brazilian_ecommerce.prod.staging_customers")

gold_delivery_by_region = (
    staging_orders
    .join(staging_customers.select("customer_id", "customer_state"), on="customer_id", how="left")
    .withColumn(
        "delivery_days",
        F.datediff(F.to_date("order_delivered_customer_date"), F.to_date("order_purchase_timestamp"))
    )
    .groupBy("customer_state")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.avg("delivery_days").alias("avg_delivery_days"),
        F.count(F.when(F.col("delivery_status").isNotNull(), 1)).alias("completed_orders"),
        F.count(F.when(F.col("delivery_status") == "delayed", 1)).alias("delayed_orders"),
    )
    .withColumn(
        "pct_delayed",
        F.when(
            F.col("completed_orders") > 0,
            (F.col("delayed_orders") / F.col("completed_orders")).cast("decimal(5,4)")
        )
    )
    .drop("delayed_orders")
    .withColumn("_loaded_at", F.current_timestamp())
    .orderBy(F.desc("total_orders"))
)

full_table_name = "brazilian_ecommerce.prod.gold_delivery_by_region"

(
    gold_delivery_by_region.write
    .mode("overwrite")
    .option("comment", "gold table: order volume, delivery speed and delay rate by customer state")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

print(f"Loaded {gold_delivery_by_region.count()} rows into {full_table_name}")